# GCSE-level school quality: a general effect and its consistency

The subject notebooks each regress A-level VA on a single GCSE element. But a school is not judged on one GCSE element: an "outstanding" school is one whose GCSE value added is **consistently high across the board**. This notebook builds that picture at GCSE level, before it is linked to A-level.

For each school we have six GCSE value-added elements, each with a published confidence interval: English, Maths, Science, Humanities, Languages and Open. We model them together with two school-level latent quantities:

- a **general GCSE quality** $g_i$: how high the school's value added is across all elements;
- a **consistency** $s_i$: how much its elements scatter around that general level (small $s_i$ means consistent, large means uneven).

Each element is measured with known error, exactly as in the A-level notebooks. We also ask whether the two are related: are the schools with the highest general quality also the most consistent?

The data contain overall Progress 8 (`P8MEA`) and an EBacc element (`P8MEAEBAC`). We do not use them. `P8MEA` is almost exactly a weighted sum of English, Maths, EBacc and Open, and the EBacc element already contains Science, Humanities and Languages, so including them would count the same information twice.

In [ ]:
import arviz as az
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import pytensor.tensor as pt
import scipy.sparse as sp

%config InlineBackend.figure_format = 'retina'
RANDOM_SEED = 8927
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-darkgrid")

print(f"Running on PyMC v{pm.__version__}")

## Data

We reshape `all-value-add-errors.csv` to long format: one row per school × GCSE element with the published value added and its confidence interval. The standard error is `(upper - lower) / (2 * 1.96)`. English, Maths and Open are available for every school with Progress 8; Science, Humanities and Languages are missing for a few schools, which simply contribute fewer rows.

In [ ]:
raw = pd.read_csv("data/all-value-add-errors.csv")

elements = ["English", "Maths", "Science", "Humanities", "Languages", "Open"]
column = {"English": "P8MEAENG", "Maths": "P8MEAMAT", "Science": "SCIVAMEA_PTQ_EE",
          "Humanities": "HUMVAMEA_PTQ_EE", "Languages": "LANVAMEA_PTQ_EE", "Open": "P8MEAOPEN"}
# Number of pupils behind each element: Science, Humanities and Languages have their own; the Progress 8 elements use P8 pupils.
pupils_column = {e: (f"{column[e]} pupils" if f"{column[e]} pupils" in raw else "P8 pupils") for e in elements}

Z_95 = 1.96
frames = []
for e in elements:
    c = column[e]
    sub = raw[["URN", c, f"{c} lower", f"{c} upper", pupils_column[e]]].dropna()
    sub.columns = ["URN", "va", "lower", "upper", "pupils"]
    sub["element"] = e
    frames.append(sub)
long = pd.concat(frames)
long["se"] = (long["upper"] - long["lower"]) / (2 * Z_95)

# Keep schools with the three Progress 8 elements (all of them have these) so every school has at least three scores.
n_elements_per_school = long.groupby("URN")["element"].nunique()
keep = n_elements_per_school[n_elements_per_school >= 3].index
long = long[long["URN"].isin(keep)].sort_values("URN").reset_index(drop=True)

urns = pd.Index(sorted(long["URN"].unique()))
long["school_idx"] = urns.get_indexer(long["URN"])
long["element_idx"] = long["element"].map({e: k for k, e in enumerate(elements)}).to_numpy()

print(f"{len(urns)} schools, {len(long)} school-element observations")
print("elements per school:", long.groupby("URN")["element"].nunique().value_counts().sort_index().to_dict())
summary = long.groupby("element")[["va", "se"]].agg(["count", "mean", "std", "median"]).round(3).loc[elements]
summary

### Checking the confidence-interval formula

DfE's interval is $\pm 1.96\,\sigma_{national}/\sqrt{n}$, so $se\sqrt{n}$ should be constant within each element.

In [ ]:
print((long["se"] * np.sqrt(long["pupils"])).groupby(long["element"]).agg(["mean", "std"]).round(3).loc[elements])

Each element has its own near-constant national SD, so the standard errors are pure sampling noise. Languages has the largest standard error (about 0.21 against 0.11-0.14 for the others) because it counts only the pupils entered for languages.

## Exploratory look

Left: the raw correlation between elements across schools. Right: the eigenvalues of that correlation matrix, for schools with all six elements. A single dominant eigenvalue means one general factor accounts for most of the shared variation.

In [ ]:
wide = long.pivot(index="URN", columns="element", values="va")[elements]
complete = wide.dropna()
raw_corr = complete.corr()
eigvals = np.sort(np.linalg.eigvalsh(raw_corr.to_numpy()))[::-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={"width_ratios": [1.3, 1]})
axes[0].imshow(raw_corr, vmin=0.3, vmax=1, cmap="Blues")
axes[0].set_xticks(range(6), elements, rotation=30)
axes[0].set_yticks(range(6), elements)
for i in range(6):
    for j in range(6):
        axes[0].text(j, i, f"{raw_corr.iloc[i, j]:.2f}", ha="center", va="center")
axes[0].set_title(f"Raw correlation of GCSE VA elements ({len(complete)} schools with all six)")
axes[0].grid(False)
axes[1].bar(range(1, 7), eigvals / eigvals.sum(), color="#4C72B0")
axes[1].set_xlabel("component")
axes[1].set_ylabel("share of variance")
axes[1].set_title("Eigenvalues of the correlation matrix")
plt.tight_layout()
plt.show()
print("share of variance in first component:", round(eigvals[0] / eigvals.sum(), 3))

TODO_EDA_NOTE

## Model

For school $i$ and element $e$ (English, Maths, Science, Humanities, Languages, Open), the observed value added $x^{obs}_{ie}$ has known standard error $\sigma_{ie}$:

$$
\begin{aligned}
x^{obs}_{ie} &\sim \text{Normal}(x_{ie}, \sigma_{ie}), \qquad x_{ie} = \mu_e + \lambda_e\, g_i + \delta_{ie}, \qquad \delta_{ie} \sim \text{Normal}(0,\ \tau_e\, s_i) \\
\log s_i &= \sigma_s\left(\rho\, g_i + \sqrt{1-\rho^2}\; w_i\right), \qquad g_i,\, w_i \sim \text{Normal}(0, 1)
\end{aligned}
$$

- $g_i$ is the school's **general quality**, on a scale of one standard deviation across schools. $\lambda_e > 0$ says how strongly element $e$ follows it and $\mu_e$ is the element's average.
- $\delta_{ie}$ is the school's **element-specific departure** from its general level. Its typical size is $\tau_e$ for an average school, multiplied by the school's consistency factor $s_i$. The prior on $\log s_i$ is centred at zero, so $s_i = 1$ is a typical school, $s_i = 2$ has twice the usual scatter and $s_i = 0.5$ half.
- $\sigma_s$ is how much schools differ in consistency. If $\sigma_s = 0$ every school is equally consistent and the model reduces to a plain one-factor model.
- $\rho$ is the correlation between general quality and $\log s_i$. $\rho < 0$ would mean the higher-quality schools are the more consistent ones.

As in the A-level factor models, we **marginalise out** $\delta_{ie}$ analytically: given $g_i$ and $s_i$, $x^{obs}_{ie} \sim \text{Normal}(\mu_e + \lambda_e g_i,\ \sqrt{\tau_e^2 s_i^2 + \sigma_{ie}^2})$. That removes a latent parameter per observation. We fit two versions: one with **constant** consistency ($s_i = 1$ for all schools) and the full model, to see whether allowing schools to differ in consistency is supported.

In [ ]:
n_schools = len(urns)
n_elements = len(elements)
x_obs = long["va"].to_numpy()
x_se = long["se"].to_numpy()
s_idx = long["school_idx"].to_numpy()
e_idx = long["element_idx"].to_numpy()

def build_model(vary_consistency):
    with pm.Model(coords={"element": elements}) as model:
        mu = pm.Normal("mu", 0, 1, dims="element")
        lam = pm.HalfNormal("lam", 1, dims="element")
        tau = pm.HalfNormal("tau", 0.5, dims="element")
        g = pm.Normal("g", 0, 1, shape=n_schools)
        if vary_consistency:
            sigma_s = pm.HalfNormal("sigma_s", 0.5)
            rho = pm.Deterministic("rho", 2 * pm.Beta("rho_raw", 2, 2) - 1)
            w = pm.Normal("w", 0, 1, shape=n_schools)
            log_s = pm.Deterministic("log_s", sigma_s * (rho * g + pt.sqrt(1 - rho**2) * w))
            specific_sd = tau[e_idx] * pt.exp(log_s[s_idx])
        else:
            specific_sd = tau[e_idx]
        pm.Normal("x_obs", mu=mu[e_idx] + lam[e_idx] * g[s_idx],
                  sigma=pt.sqrt(specific_sd**2 + x_se**2), observed=x_obs)
    return model

constant_model = build_model(vary_consistency=False)
full_model = build_model(vary_consistency=True)

### Prior predictive check

In [ ]:
with full_model:
    prior = pm.sample_prior_predictive(draws=300, random_seed=RANDOM_SEED)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.hist(prior.prior_predictive["x_obs"].to_numpy().ravel(), bins=60, color="#4C72B0", range=(-6, 6))
ax.set_title("Prior predictive: GCSE element VA")
plt.tight_layout()
plt.show()
print("observed VA range:", x_obs.min().round(2), "to", x_obs.max().round(2))
del prior

### Fit

In [ ]:
with constant_model:
    idata_const = pm.sample(draws=1000, tune=2000, chains=4, target_accept=0.99,
                            random_seed=RANDOM_SEED, progressbar=False)

In [ ]:
with full_model:
    idata_full = pm.sample(draws=1000, tune=2000, chains=4, target_accept=0.99,
                           random_seed=RANDOM_SEED, progressbar=False)

### Diagnostics

In [ ]:
for name, idata in [("constant consistency", idata_const), ("varying consistency", idata_full)]:
    print(f"{name}: divergences = {int(idata.sample_stats['diverging'].sum())}")
az.summary(idata_full, var_names=["mu", "lam", "tau", "sigma_s", "rho"], round_to=3)

## What the general factor shows

### How strongly does each element follow general quality?

$\lambda_e$ is the change in element $e$'s value added for a one-SD-higher $g_i$. The share of an element's between-school variance that is general is $\lambda_e^2 / (\lambda_e^2 + \tau_e^2 + \bar\sigma_e^2)$, where the last term is the typical sampling variance. The remainder is either element-specific departures from general quality or sampling noise.

In [ ]:
post = idata_full.posterior
pc = idata_const.posterior

def summarise(x):
    lo, hi = np.percentile(x, [5.5, 94.5])
    return f"mean={x.mean():.3f}, 89% interval [{lo:.3f}, {hi:.3f}]"

mean_se2 = long.groupby("element")["se"].apply(lambda s: (s**2).mean()).reindex(elements).to_numpy()
rows = []
for k, e in enumerate(elements):
    lam_k = post["lam"].sel(element=e).to_numpy().ravel()
    tau_k = post["tau"].sel(element=e).to_numpy().ravel()
    share_general = lam_k**2 / (lam_k**2 + tau_k**2 + mean_se2[k])
    share_noise = mean_se2[k] / (lam_k**2 + tau_k**2 + mean_se2[k])
    rows.append({"element": e, "loading_lam": lam_k.mean(), "specific_sd_tau": tau_k.mean(),
                 "typical_se": np.sqrt(mean_se2[k]), "share_general": share_general.mean(),
                 "share_specific": 1 - share_general.mean() - share_noise.mean(), "share_noise": share_noise.mean()})
loadings = pd.DataFrame(rows).set_index("element").round(3)
display(loadings)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for k, e in enumerate(elements):
    for ax, var, label in [(axes[0], "lam", "loading on general quality, $\\lambda_e$"), (axes[1], "tau", "typical element-specific SD, $\\tau_e$")]:
        lo, m, hi = np.percentile(post[var].sel(element=e).to_numpy().ravel(), [5.5, 50, 94.5])
        ax.plot([lo, hi], [k, k], color="#4C72B0", linewidth=2)
        ax.plot(m, k, "o", color="#4C72B0")
        ax.set_xlabel(label)
for ax in axes:
    ax.set_yticks(range(6), elements)
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Do schools really differ in consistency?

The full model lets each school have its own consistency $s_i$. If schools were all equally consistent, $\sigma_s$ would be near zero. Below: the posterior of $\sigma_s$ and of $\rho$ (the link between general quality and consistency).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(post["sigma_s"].to_numpy().ravel(), bins=40, density=True, color="#4C72B0")
axes[0].set_xlabel("sigma_s: spread of log consistency across schools")
axes[1].hist(post["rho"].to_numpy().ravel(), bins=40, density=True, color="#DD8452")
axes[1].axvline(0, color="grey", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("rho: correlation of general quality with log s")
plt.tight_layout()
plt.show()
print("sigma_s:", summarise(post["sigma_s"].to_numpy().ravel()))
print("rho:    ", summarise(post["rho"].to_numpy().ravel()))
sig = post["sigma_s"].to_numpy().ravel()
print(f"a school one SD above/below average in log consistency has element-specific scatter x{np.exp(sig.mean()):.2f} / x{np.exp(-sig.mean()):.2f} the typical school's")

### A posterior predictive check: does constant consistency fit?

For each school and posterior draw we compute how surprising its six (or fewer) element scores are relative to the model, $T_i = \frac{1}{k_i}\sum_e r_{ie}^2$ with $r_{ie} = (x^{obs}_{ie} - \mu_e - \lambda_e g_i)/\sqrt{\tau_e^2 s_i^2 + \sigma_{ie}^2}$. Data that fit the model give $T_i$ near 1, and if some schools are more scattered than the model allows we see too many large $T_i$. We compare the observed share of schools above a few thresholds with the share in data replicated from the model, for the constant-consistency model and the full model.

In [ ]:
counts = np.bincount(s_idx, minlength=n_schools).astype(float)
school_matrix = sp.csr_matrix((np.ones(len(s_idx)), (s_idx, np.arange(len(s_idx)))), shape=(n_schools, len(s_idx)))
thin = slice(0, 1000, 5)   # 200 draws per chain, 800 in total

def draw_array(p, name, extra_dims=1):
    arr = p[name].to_numpy()[:, thin]
    return arr.reshape(-1, *arr.shape[2:])

def ppc_share(p, vary, thresholds=(1.5, 2.0, 3.0)):
    mu_d, lam_d, tau_d, g_d = (draw_array(p, n) for n in ["mu", "lam", "tau", "g"])
    n_draws = len(mu_d)
    log_s_d = draw_array(p, "log_s") if vary else np.zeros((n_draws, n_schools))
    mean = mu_d[:, e_idx] + lam_d[:, e_idx] * g_d[:, s_idx]
    sd = np.sqrt((tau_d[:, e_idx] * np.exp(log_s_d[:, s_idx])) ** 2 + x_se**2)
    rep = mean + sd * rng.standard_normal(mean.shape)
    out = {}
    for label, values in [("observed", x_obs[None, :]), ("replicated", rep)]:
        r2 = ((values - mean) / sd) ** 2
        T = (school_matrix @ r2.T).T / counts
        out[label] = np.stack([(T > t).mean(axis=1) for t in thresholds], axis=1)
    return out

rows = []
for name, p, vary in [("constant consistency", pc, False), ("varying consistency", post, True)]:
    res = ppc_share(p, vary)
    for j, t in enumerate((1.5, 2.0, 3.0)):
        lo, hi = np.percentile(res["replicated"][:, j], [5.5, 94.5])
        rows.append({"model": name, "threshold_T": t, "observed_share": res["observed"][:, j].mean(),
                     "replicated_mean": res["replicated"][:, j].mean(), "replicated_89%": f"[{lo:.3f}, {hi:.3f}]"})
pd.DataFrame(rows).round(3)

TODO_PPC_NOTE

## Which schools are outstanding, and consistently so?

Each school has a posterior for $g_i$ and $\log s_i$. The scatter shows the posterior means, coloured by how many elements the school has. The shared-scale summaries below tell us how well an individual school's values are pinned down: a posterior SD near the prior SD (1 for $g_i$, $\sigma_s$ for $\log s_i$) means the data taught us little about that school.

In [ ]:
g_post = post["g"].to_numpy().reshape(-1, n_schools)
log_s_post = post["log_s"].to_numpy().reshape(-1, n_schools)
g_mean, g_sd = g_post.mean(axis=0), g_post.std(axis=0)
ls_mean, ls_sd = log_s_post.mean(axis=0), log_s_post.std(axis=0)
n_el = counts.astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc = axes[0].scatter(g_mean, np.exp(ls_mean), c=n_el, cmap="viridis", s=8, alpha=0.6)
axes[0].axhline(1, color="grey", linewidth=0.8, linestyle="--")
axes[0].axvline(0, color="grey", linewidth=0.8, linestyle="--")
axes[0].set_yscale("log")
axes[0].set_xlabel("general GCSE quality, $g_i$ (posterior mean)")
axes[0].set_ylabel("consistency factor $s_i$ (posterior mean; lower = more consistent)")
plt.colorbar(sc, ax=axes[0], label="number of GCSE elements")
axes[1].scatter(g_mean, g_sd, s=6, alpha=0.4, color="#4C72B0")
axes[1].set_xlabel("$g_i$ (posterior mean)")
axes[1].set_ylabel("posterior SD of $g_i$")
axes[1].set_ylim(0, 1)
plt.show()

sigma_s_mean = post["sigma_s"].to_numpy().mean()
print(f"posterior SD of g_i: median {np.median(g_sd):.2f} (prior 1)")
print(f"posterior SD of log s_i: median {np.median(ls_sd):.2f} (prior {sigma_s_mean:.2f}, i.e. the posterior mean of sigma_s)")
print("correlation of posterior means, g vs log s:", np.corrcoef(g_mean, ls_mean)[0, 1].round(3))

TODO_SCHOOLS_NOTE

## Is one general factor enough? A check on leftover structure

The raw correlations suggested the pairs Maths-Science and English-Open are slightly more alike than a single factor implies. If so, "consistency" as defined above would partly reflect a systematic split (for example maths-and-science schools versus language-and-open schools) instead of random scatter. We compute the standardised residuals $r_{ie}$ from the full model and their correlation across schools (averaged over posterior draws). Under a correct one-factor model these should be uncorrelated, apart from a small negative correlation created by estimating $g_i$ from the same data.

In [ ]:
mu_d, lam_d, tau_d = (draw_array(post, n) for n in ["mu", "lam", "tau"])
g_d, log_s_d = draw_array(post, "g"), draw_array(post, "log_s")
mean = mu_d[:, e_idx] + lam_d[:, e_idx] * g_d[:, s_idx]
r = (x_obs[None, :] - mean) / np.sqrt((tau_d[:, e_idx] * np.exp(log_s_d[:, s_idx])) ** 2 + x_se**2)

resid_wide = pd.DataFrame({"school": s_idx, "element": e_idx, "r": r.mean(axis=0)}).pivot(index="school", columns="element", values="r")
resid_wide.columns = elements
resid_corr = resid_wide.dropna().corr()

fig, ax = plt.subplots(figsize=(6.2, 5))
im = ax.imshow(resid_corr, vmin=-0.5, vmax=0.5, cmap="RdBu_r")
ax.set_xticks(range(6), elements, rotation=30)
ax.set_yticks(range(6), elements)
for i in range(6):
    for j in range(6):
        ax.text(j, i, f"{resid_corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
ax.set_title("Correlation of standardised residuals")
ax.grid(False)
plt.colorbar(im, ax=ax)
plt.show()

TODO_RESID_NOTE

## Regional pattern (descriptive)

The mean posterior general quality and consistency by region, with the number of schools. This is descriptive, not a separate model: the shrinkage in $g_i$ and $\log s_i$ already pulls poorly determined schools towards the overall average.

In [ ]:
region = pd.Series(urns.map(raw.drop_duplicates("URN").set_index("URN")["RGN24NM"]), index=range(n_schools))
by_region = pd.DataFrame({"g": g_mean, "s": np.exp(ls_mean), "region": region}).groupby("region").agg(
    schools=("g", "count"), mean_g=("g", "mean"), sd_g=("g", "std"), mean_s=("s", "mean")).sort_values("schools", ascending=False)
display(by_region.round(3))

order = list(by_region.index)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), sharey=True)
data_g = [g_mean[(region == r_).to_numpy()] for r_ in order]
data_s = [np.exp(ls_mean[(region == r_).to_numpy()]) for r_ in order]
axes[0].boxplot(data_g, vert=False, tick_labels=order, showfliers=False)
axes[0].axvline(0, color="grey", linewidth=0.8, linestyle="--")
axes[0].set_xlabel("general quality $g_i$ (posterior mean)")
axes[1].boxplot(data_s, vert=False, tick_labels=order, showfliers=False)
axes[1].axvline(1, color="grey", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("consistency factor $s_i$ (posterior mean)")
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()

TODO_INTERP